[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Showmick119/Fine-Tuning-Open-Source-LLM/blob/main/notebooks/finetune_code_llama.ipynb)

# CodeLlama-7B Fine-Tuning For FastAPI Code Generation

This notebook fine-tunes CodeLlama-7b-Instruct-hf specifically for FastAPI code generation using QLoRA (4-bit quantization + LoRA).

## Prerequisites

1. **HuggingFace Account**: Accept CodeLlama terms at https://huggingface.co/codellama/CodeLlama-7b-Instruct-hf
2. **API Tokens**: HuggingFace token and OpenAI API key for evaluation
3. **GPU**: T4 or better (select GPU runtime in Colab)
4. **Estimated Runtime**: 1-2 hours for complete fine-tuning and evaluation

## Expected Improvements

Our baseline results showed improvement areas in:
- Pydantic models and type hints
- Error handling patterns
- Input validation
- Documentation and response models

Goal: Improve average score from ~ 69% to 85%+

## 1️⃣ Setup & Dependencies

Install required packages and verify GPU availability.

In [ ]:
print("Installing required packages...")
print("Using Colab-compatible versions")
print("\n" + "="*50)

%pip install -q transformers accelerate peft datasets trl bitsandbytes huggingface_hub openai

print("Installation complete!")

print("\n→ Testing package imports...")
try:
    import torch
    import transformers
    import accelerate
    import peft
    import datasets
    import trl
    import bitsandbytes as bnb
    import numpy as np
    
    print(f"✓ NumPy: {np.__version__}")
    print(f"✓ PyTorch: {torch.__version__}")
    print(f"✓ Transformers: {transformers.__version__}")
    print(f"✓ Bitsandbytes: {bnb.__version__}")
    print(f"✓ PEFT: {peft.__version__}")
    
    if torch.cuda.is_available():
        print(f"✓ CUDA: {torch.version.cuda}")
        print(f"✓ GPU: {torch.cuda.get_device_name(0)}")

        from transformers import BitsAndBytesConfig
        test_config = BitsAndBytesConfig(load_in_4bit=True)
        print("QUANTIZATION WORKS!")
    
    print("\nALL PACKAGES READY!")
    
except Exception as e:
    print(f"Warning: {e}")
    print("Trying to continue anyway...")

print("\n" + "="*50)
print("SETUP COMPLETE - READY FOR FINE-TUNING!")
print("="*50)

In [ ]:
import torch
import os
import json
from pathlib import Path
import logging
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

try:
    import transformers
    import accelerate
    import peft
    import datasets
    import trl
    import bitsandbytes
    print("✓ All required packages imported successfully")

    print(f"PyTorch: {torch.__version__}")
    print(f"Transformers: {transformers.__version__}")
    print(f"Bitsandbytes: {bitsandbytes.__version__}")
    
except ImportError as e:
    print(f"✗ Package import failed: {e}")
    print("Please run the previous cell again to fix package installation")

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("\nGPU Status:")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    cuda_version = torch.version.cuda
    print(f"GPU: {gpu_name}")
    print(f"Memory: {gpu_memory:.1f} GB")
    print(f"PyTorch CUDA version: {cuda_version}")
    
    if gpu_memory < 15:
        print("Warning: Less than 15GB GPU memory. Consider using smaller batch sizes.")
else:
    print("No GPU detected! Please enable GPU in Runtime > Change runtime type")
    raise RuntimeError("GPU required for fine-tuning")

print("\nSetup complete! Ready to proceed with model loading.")

## 2️⃣ Repository Setup

Clone the repository and set up Python path for importing modules.

In [ ]:
import os
import sys
from pathlib import Path

if 'google.colab' in sys.modules:
    print("Running in Google Colab - setting up repository...")

    if not Path('/content/Fine-Tuning-Open-Source-LLM').exists():
        !git clone https://github.com/Showmick119/Fine-Tuning-Open-Source-LLM.git /content/Fine-Tuning-Open-Source-LLM
    
    os.chdir('/content/Fine-Tuning-Open-Source-LLM')

    project_root = '/content/Fine-Tuning-Open-Source-LLM'
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
        
    print(f"Repository cloned and set up at: {project_root}")
else:
    print("Running locally - using existing repository")
    project_root = Path.cwd()

print(f"Working directory: {os.getcwd()}")
print(f"Python path includes: {project_root}")

required_files = [
    'model/load_base_model.py',
    'data/prepare_dataset.py', 
    'train/run_lora_finetune.py',
    'evaluate/fastapi_evaluator.py',
    'evaluate/llm_judge.py',
    'data/data/fastapi_mined_dataset.json'
]

missing_files = [f for f in required_files if not Path(f).exists()]
if missing_files:
    print(f"Missing files: {missing_files}")
    raise FileNotFoundError("Required codebase files not found")
else:
    print("All required codebase files found")

## 3️⃣ Authentication Setup

Configure HuggingFace and OpenAI API credentials.

In [ ]:
from huggingface_hub import login
from google.colab import userdata
import openai

print("Setting up authentication...")

hf_token = os.getenv('HF_TOKEN')
if not hf_token:
    if 'google.colab' in sys.modules:
        try:
            hf_token = userdata.get('HF_TOKEN')
        except:
            pass
    
    if not hf_token:
        print("Please enter your HuggingFace token:")
        print("(Get it from: https://huggingface.co/settings/tokens)")
        hf_token = input("HF Token: ").strip()

login(token=hf_token)
os.environ['HF_TOKEN'] = hf_token
print("HuggingFace authentication successful")

openai_key = os.getenv('OPENAI_API_KEY')
if not openai_key:
    if 'google.colab' in sys.modules:
        try:
            openai_key = userdata.get('OPENAI_API_KEY')
        except:
            pass
    
    if not openai_key:
        print("Please enter your OpenAI API key for evaluation:")
        openai_key = input("OpenAI API Key: ").strip()

os.environ['OPENAI_API_KEY'] = openai_key
openai.api_key = openai_key
print("OpenAI API key configured for evaluation")

## 4️⃣ Model Loading & Configuration

Load CodeLlama-7b base model with quantization configuration using our modular loader.

In [ ]:
from model.load_base_model import ModelLoader

print("Loading CodeLlama-7b with QLoRA configuration...")

model_loader = ModelLoader("configs/lora_config.json")

try:
    base_model, tokenizer = model_loader.load_model_and_tokenizer()
    print("Model and tokenizer loaded successfully!")

    total_params = sum(p.numel() for p in base_model.parameters())
    trainable_params = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
    
    print(f"Model Statistics:")
    print(f"  • Total parameters: {total_params:,}")
    print(f"  • Trainable parameters: {trainable_params:,}")
    print(f"  • Trainable ratio: {100 * trainable_params / total_params:.2f}%")
    
    if torch.cuda.is_available():
        memory_used = torch.cuda.memory_allocated() / 1e9
        print(f"  • GPU memory used: {memory_used:.2f} GB")
        
except Exception as e:
    print(f"Error loading model: {e}")
    raise

## 5️⃣ Dataset Preparation

Load and preprocess the FastAPI dataset using our custom dataset preparator.

In [ ]:
from data.prepare_dataset import DatasetPreparator

print("Preparing FastAPI training dataset...")

dataset_preparator = DatasetPreparator(
    dataset_path="data/data/fastapi_mined_dataset.json"
)

dataset_preparator.tokenizer = tokenizer
dataset_preparator.max_length = 512

try:
    raw_dataset = dataset_preparator.load_and_prepare_dataset()
    print(f"Loaded {len(raw_dataset)} training examples")

    categories = {}
    difficulties = {}
    for example in raw_dataset:
        if 'category' in example:
            categories[example['category']] = categories.get(example['category'], 0) + 1
        if 'difficulty' in example:
            difficulties[example['difficulty']] = difficulties.get(example['difficulty'], 0) + 1
    
    print("Dataset Statistics:")
    if categories:
        print("  Categories:")
        for cat, count in sorted(categories.items()):
            print(f"    • {cat}: {count} examples")
    
    if difficulties:
        print("  Difficulty levels:")
        for diff, count in sorted(difficulties.items()):
            print(f"    • {diff}: {count} examples")

    print("\\nSample training example:")
    sample = raw_dataset[0]
    print(f"  Instruction: {sample['instruction']}")
    print(f"  Output: {sample['output']}")
    
except Exception as e:
    print(f"Error preparing dataset: {e}")
    raise

## 6️⃣ Training Configuration

Configure training parameters and prepare for fine-tuning.

In [ ]:
from train.run_lora_finetune import load_config

print("Loading training configuration from JSON files...")

TRAINING_CONFIG = load_config("configs/training_args.json")
LORA_CONFIG = load_config("configs/lora_config.json") 
HUB_CONFIG = load_config("configs/hub_config.json")

hub_model_id = f"{HUB_CONFIG['hub_model_id_prefix']}-{datetime.now().strftime('%Y%m%d')}"

print("Configuration loaded from JSON files:")
print(f"  • Training config: configs/training_args.json")
print(f"  • LoRA config: configs/lora_config.json")
print(f"  • Hub config: configs/hub_config.json")

print("\\nTraining parameters:")
print(f"  • Epochs: {TRAINING_CONFIG['num_train_epochs']}")
print(f"  • Batch size: {TRAINING_CONFIG['per_device_train_batch_size']}")
print(f"  • Gradient accumulation: {TRAINING_CONFIG['gradient_accumulation_steps']}")
print(f"  • Effective batch size: {TRAINING_CONFIG['per_device_train_batch_size'] * TRAINING_CONFIG['gradient_accumulation_steps']}")
print(f"  • Learning rate: {TRAINING_CONFIG['learning_rate']}")

print("\\nLoRA parameters:")
print(f"  • r: {LORA_CONFIG['lora_config']['r']}")
print(f"  • alpha: {LORA_CONFIG['lora_config']['lora_alpha']}")
print(f"  • dropout: {LORA_CONFIG['lora_config']['lora_dropout']}")

print("\\nHub configuration:")
print(f"  • Push to hub: {HUB_CONFIG['push_to_hub']}")
print(f"  • Model ID: {hub_model_id}")

output_dir = Path("./results")
output_dir.mkdir(exist_ok=True)
print(f"  • Output directory: {output_dir.absolute()}")

## 7️⃣ Fine-tuning Process

Run the QLoRA fine-tuning using our modular training infrastructure.

In [ ]:
from train.run_lora_finetune import run_training
import traceback

print("Starting LoRA fine-tuning process...")

try:
    processed_dataset = dataset_preparator.prepare_dataset()
    print(f"Processed dataset with {len(processed_dataset)} examples")
    
except Exception as e:
    print(f"Error processing dataset: {e}")
    raise

print("\nBeginning fine-tuning...")
print("This will take approximately 2-4 hours depending on your GPU...")

training_start_time = datetime.now()

try:
    finetuned_model = run_training(
        model=base_model,
        tokenizer=tokenizer,
        dataset=processed_dataset,
        training_config_path="configs/training_args.json",
        hub_config_path="configs/hub_config.json",
        output_dir="./results"
    )
    
    training_end_time = datetime.now()
    training_duration = training_end_time - training_start_time
    
    print(f"\nFine-tuning completed!")
    print(f"Training duration: {training_duration}")
    print(f"Model pushed to HuggingFace Hub: {hub_model_id}")

    if torch.cuda.is_available():
        final_memory = torch.cuda.memory_allocated() / 1e9
        max_memory = torch.cuda.max_memory_allocated() / 1e9
        print(f"Final GPU memory: {final_memory:.2f} GB")
        print(f"Peak GPU memory: {max_memory:.2f} GB")
        
except Exception as e:
    print(f"Training failed: {e}")
    traceback.print_exc()
    raise

## 8️⃣ Model Evaluation & Comparison

Compare the fine-tuned model with the base model using our evaluation framework.

In [ ]:
from evaluate.fastapi_evaluator import FastAPIEvaluator
from evaluate.llm_judge import GPTFastAPIJudge, format_gpt_evaluation

print("Setting up evaluation framework...")

fastapi_evaluator = FastAPIEvaluator()
gpt_judge = GPTFastAPIJudge()

TEST_CASES = [
    {
        "name": "Dataset Example - Project List Endpoint",
        "prompt": "Create a FastAPI GET endpoint that lists projects for an organization. It should: 1) Accept organization_id as a parameter, 2) Use database session from dependencies, 3) Return a list of projects, 4) Include proper error handling if no projects found. Return ONLY the code without any explanation.",
    },
    {
        "name": "Simple Hello World",
        "prompt": "Create a FastAPI GET endpoint that returns 'Hello, World!'. Return ONLY the code without any explanation.",
    },
    {
        "name": "JSON Payload Handler", 
        "prompt": "Create a FastAPI POST endpoint that accepts a JSON payload with 'name' and 'age' fields and returns a greeting. Return ONLY the code without any explanation.",
    },
    {
        "name": "Error Handling",
        "prompt": "Create a FastAPI endpoint that demonstrates proper error handling with HTTPException. Return ONLY the code without any explanation.",
    },
    {
        "name": "Path Parameters",
        "prompt": "Create a FastAPI endpoint that accepts a user_id as a path parameter and returns user information. Return ONLY the code without any explanation.",
    }
]

print(f"Evaluation setup complete with {len(TEST_CASES)} test cases")

In [ ]:
def generate_response(model, tokenizer, prompt, max_new_tokens=512) -> str:
    """
    Generate response from the model.

    Args:
        model
        tokenizer
        prompt
        max_new_tokens
    
    Returns:
        str: Response generated by the Language Model
    """
    formatted_prompt = f"[INST] {prompt} [/INST]"
    inputs = tokenizer(formatted_prompt, return_tensors="pt", padding=True)

    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
        
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response[len(formatted_prompt):].strip()

print("Running comprehensive evaluation...")
print("\\nTesting fine-tuned model vs baseline expectations...")

evaluation_results = []
fastapi_scores = []
gpt_scores = []

for i, test_case in enumerate(TEST_CASES, 1):
    print(f"\\n{'='*80}")
    print(f"Test {i}/{len(TEST_CASES)}: {test_case['name']}")
    print(f"{'='*80}")

    response = generate_response(finetuned_model, tokenizer, test_case['prompt'])
    
    print("\\nGenerated Code:")
    print("-" * 50)
    print(response)
    print("-" * 50)

    fastapi_result = fastapi_evaluator.evaluate_response(test_case['prompt'], response)
    fastapi_score = fastapi_result.score * 100
    fastapi_scores.append(fastapi_score)
    
    print("\\nFastAPI Evaluator Results:")
    print(fastapi_evaluator.format_evaluation_result(fastapi_result))

    gpt_result = gpt_judge.evaluate(test_case['prompt'], response)
    gpt_score = gpt_result.total_score
    gpt_scores.append(gpt_score)
    
    print("\\nGPT Judge Results:")
    print(format_gpt_evaluation(gpt_result))

    evaluation_results.append({
        'test_name': test_case['name'],
        'prompt': test_case['prompt'],
        'response': response,
        'fastapi_score': fastapi_score,
        'gpt_score': gpt_score,
        'fastapi_result': fastapi_result,
        'gpt_result': gpt_result
    })

print(f"\\n{'='*80}")
print("EVALUATION SUMMARY")
print(f"{'='*80}")

avg_fastapi = sum(fastapi_scores) / len(fastapi_scores)
avg_gpt = sum(gpt_scores) / len(gpt_scores)

print(f"\\nAverage Scores:")
print(f"  • FastAPI Evaluator: {avg_fastapi:.1f}/100")
print(f"  • GPT Judge: {avg_gpt:.1f}/100") 

print(f"\\nIndividual Test Scores:")
for i, result in enumerate(evaluation_results):
    print(f"  {i+1}. {result['test_name'][:30]:<30} FastAPI: {result['fastapi_score']:.1f} | GPT: {result['gpt_score']:.1f}")

## 9️⃣ Save Results

Save evaluation results and model artifacts for future reference.

In [ ]:
import shutil
from google.colab import drive

print("Saving evaluation results...")

results_summary = {
    "model_info": {
        "base_model": "codellama/CodeLlama-7b-Instruct-hf",
        "finetuned_model": hub_model_id,
        "training_duration": str(training_duration),
        "training_config": TRAINING_CONFIG,
        "lora_config": LORA_CONFIG,
        "hub_config": HUB_CONFIG,
        "evaluation_date": datetime.now().isoformat()
    },
    "evaluation_scores": {
        "average_fastapi_score": avg_fastapi,
        "average_gpt_score": avg_gpt,
        "individual_scores": [
            {
                "test_name": result["test_name"],
                "fastapi_score": result["fastapi_score"],
                "gpt_score": result["gpt_score"]
            }
            for result in evaluation_results
        ]
    }
}

results_file = Path("results") / f"evaluation_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
results_file.parent.mkdir(exist_ok=True)

with open(results_file, 'w') as f:
    json.dump(results_summary, f, indent=2, default=str)

print(f"Results saved to: {results_file}")

responses_file = Path("results") / f"generated_responses_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

with open(responses_file, 'w') as f:
    f.write("FINE-TUNED CODELLAMA FASTAPI EVALUATION RESULTS\\n")
    f.write("=" * 60 + "\\n\\n")
    
    for i, result in enumerate(evaluation_results, 1):
        f.write(f"TEST {i}: {result['test_name']}\\n")
        f.write("-" * 40 + "\\n")
        f.write(f"PROMPT: {result['prompt']}\\n\\n")
        f.write(f"GENERATED CODE:\\n{result['response']}\\n\\n")
        f.write(f"FASTAPI SCORE: {result['fastapi_score']:.1f}/100\\n")
        f.write(f"GPT SCORE: {result['gpt_score']:.1f}/100\\n")
        f.write("\\n" + "=" * 60 + "\\n\\n")

print(f"Detailed responses saved to: {responses_file}")

if 'google.colab' in sys.modules:
    try:
        drive.mount('/content/drive')

        drive_backup = Path('/content/drive/MyDrive/CodeLlama_FastAPI_Results')
        drive_backup.mkdir(exist_ok=True)
        
        shutil.copy2(results_file, drive_backup)
        shutil.copy2(responses_file, drive_backup)
        
        print(f"Results backed up to Google Drive: {drive_backup}")
    except Exception as e:
        print(f"Could not backup to Google Drive: {e}")

print("\\nAll results saved successfully!")

## 🔟 Interactive Testing

Test the fine-tuned model with custom prompts.

In [ ]:
def test_custom_prompt(prompt, evaluate_with_tools=True):
    """
    Test a custom prompt and optionally evaluate it.
    """
    print(f"\\nTesting Custom Prompt")
    print("=" * 50)
    print(f"Prompt: {prompt}")
    print("=" * 50)

    response = generate_response(finetuned_model, tokenizer, prompt)
    
    print("\\nGenerated Code:")
    print("-" * 30)
    print(response)
    print("-" * 30)
    
    if evaluate_with_tools:
        try:
            fastapi_result = fastapi_evaluator.evaluate_response(prompt, response)
            print(f"\\nFastAPI Score: {fastapi_result.score * 100:.1f}/100")
            
            if fastapi_result.score < 0.7:
                print("\\nKey Issues Found:")
                if not fastapi_result.is_valid_python:
                    print("  • Invalid Python syntax")
                if not fastapi_result.has_imports:
                    print("  • Missing FastAPI imports")
                if not fastapi_result.has_endpoint:
                    print("  • No endpoint decorator found")
                if not fastapi_result.has_error_handling:
                    print("  • Consider adding error handling")
                    
        except Exception as e:
            print(f"\\nEvaluation error: {e}")
    
    return response

# Example test prompts - feel free to modify
example_prompts = [
    "Create a FastAPI endpoint for user authentication with JWT tokens",
    "Create a FastAPI POST endpoint that uploads files with validation",
    "Create a FastAPI WebSocket endpoint for real-time chat",
    "Create a FastAPI endpoint with database pagination using SQLAlchemy",
    "Create a FastAPI endpoint that implements rate limiting"
]

print("Interactive Testing Environment Ready!")
print("\\nExample prompts to try:")
for i, prompt in enumerate(example_prompts, 1):
    print(f"  {i}. {prompt}")

print("\\n" + "=" * 80)
print("Usage: test_custom_prompt('Your FastAPI prompt here')")
print("=" * 80)

## Conclusion & Summary

Fine-tuning process completed! Here's what we accomplished:

In [ ]:
print("FINE-TUNING COMPLETED SUCCESSFULLY!")
print("=" * 80)

print("\\nSummary:")
print(f"  • Base Model: codellama/CodeLlama-7b-Instruct-hf")
print(f"  • Fine-tuned Model: {hub_model_id}")
print(f"  • Training Duration: {training_duration}")
print(f"  • Dataset Size: {len(raw_dataset)} examples")
print(f"  • GPU Used: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

print("\\nPerformance:")
print(f"  • Average FastAPI Score: {avg_fastapi:.1f}/100")
print(f"  • Average GPT Score: {avg_gpt:.1f}/100")

print("\\nNext Steps:")
print("  1. Test the model with your own FastAPI prompts")
print("  2. Use the saved model ID for production inference")
print("  3. Consider further fine-tuning with domain-specific data")
print("  4. Monitor performance on real-world tasks")

print("\\nResources:")
print(f"  • Model on HuggingFace Hub: https://huggingface.co/{hub_model_id}")
print(f"  • Evaluation Results: {results_file}")
print(f"  • Generated Responses: {responses_file}")

print("\\nHappy coding with your fine-tuned FastAPI assistant!")
print("=" * 80)